# 02 · Acquire financial facts — the evaluation oracle

> **Run order.** This notebook is step 2 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


This is the most important acquisition step in the project.

Every row written here is a value we know to be true. Notebook 06 searches the
parsed annual reports for these values to auto-label *which page contains the
answer*, which is how the benchmark gets built without hand-writing hundreds of
questions.

**The trap:** `financialCurrency` is **not** the quote currency. Infosys quotes
in INR on the NSE and reports its statements in USD. Inferring currency from the
exchange suffix makes Infosys look ~85× smaller than Wipro and would make the
benchmark search for `1,928` in a document printing `₹1,62,990 crore`.

In [ ]:
# The project is installed in editable mode by `uv sync`, so `analyst` imports
# directly. Nothing here manipulates sys.path.
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

import yfinance as yf
from analyst.corpus import load_corpus

companies = load_corpus()

rows = []
for c in companies:
    info = yf.Ticker(c.yf_symbol).info
    rows.append({
        "ticker": c.ticker,
        "quote_currency": info.get("currency"),
        "financialCurrency": info.get("financialCurrency"),
    })
ccy = pd.DataFrame(rows)
print("Companies whose reporting currency differs from their quote currency:")
print(ccy[ccy.quote_currency != ccy.financialCurrency].to_string(index=False))
ccy

## Wide statements to long facts

`melt_statement` drops NaN cells rather than writing `0`. An absence of evidence is not a zero, and a fabricated zero would corrupt the oracle.

In [ ]:
from analyst.facts import STATEMENT_ATTRS, melt_statement

t = yf.Ticker("SUNPHARMA.NS")
income = t.income_stmt
print(f"income_stmt shape: {income.shape}  (concepts x fiscal periods)")
income.head(6).iloc[:, :4]

In [ ]:
sample = melt_statement(income, "SUNPHARMA", "income_statement", "INR")
print(f"{len(sample)} facts from that one statement\n")
pd.DataFrame([r.model_dump() for r in sample]).head(8)

## Ingest every company

In [ ]:
from sqlalchemy import update
from sqlalchemy.dialects.postgresql import insert
from tenacity import retry, stop_after_attempt, wait_exponential

from analyst.db import session_scope
from analyst.facts import FactRow
from analyst.models import Company, Fact

CHUNK = 500

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def fetch_statements(symbol: str):
    t = yf.Ticker(symbol)
    frames = {name: getattr(t, attr, None) for name, attr in STATEMENT_ATTRS.items()}
    currency = str(t.info.get("financialCurrency") or "INR").upper()
    return frames, currency

report = []
for c in companies:
    frames, currency = fetch_statements(c.yf_symbol)
    with session_scope() as s:
        s.execute(
            update(Company).where(Company.ticker == c.ticker)
            .values(financial_currency=currency)
        )
    rows: list[FactRow] = []
    for statement, df in frames.items():
        rows.extend(melt_statement(df, c.ticker, statement, currency))
    if not rows:
        continue
    with session_scope() as s:
        for i in range(0, len(rows), CHUNK):
            stmt = insert(Fact).values([r.model_dump() for r in rows[i : i + CHUNK]])
            s.execute(stmt.on_conflict_do_update(
                constraint="uq_facts_identity",
                set_={"value": stmt.excluded.value, "unit": stmt.excluded.unit},
            ))
    periods = sorted({r.period_end for r in rows})
    report.append({
        "ticker": c.ticker, "currency": currency, "facts": len(rows),
        "concepts": len({r.concept for r in rows}),
        "from": periods[0], "to": periods[-1],
    })

df = pd.DataFrame(report)
print(f"ORACLE SIZE: {df['facts'].sum():,} facts")
df

## The oracle is real

Revenue in reporting currency, per sector. Watch INFY — it is in USD, and the table says so rather than pretending otherwise.

In [ ]:
from sqlalchemy import text
from analyst.db import session_scope

SQL = '''
SELECT c.sector, c.ticker, c.financial_currency AS ccy,
       round(f26.value / 1e7) AS fy26_cr,
       round(f25.value / 1e7) AS fy25_cr,
       round(100.0 * (f26.value - f25.value) / f25.value, 1) AS yoy_pct
FROM companies c
JOIN facts f26 ON f26.ticker = c.ticker AND f26.concept = 'Total Revenue'
              AND f26.period_end = DATE '2026-03-31'
JOIN facts f25 ON f25.ticker = c.ticker AND f25.concept = 'Total Revenue'
              AND f25.period_end = DATE '2025-03-31'
ORDER BY c.sector, yoy_pct DESC
'''
with session_scope() as s:
    out = pd.DataFrame(s.execute(text(SQL)).mappings().all())
out